# GenAI Evaluation with mlflow

## Environment Set-up

Create a virtual environment:

```
python -m venv .venv
```

Activate the virtual environment:

```
source .venv/bin/activate
```

Install the dependencies

```
pip install -r requirements.txt
```

Start mlflow server:

```
mlflow server --host 0.0.0.0 --port 2000
```

In [1]:
# Connect to mlflow server and set experiment

import mlflow

mlflow.set_tracking_uri("http://localhost:2000")
mlflow.set_experiment("GenAI Evaluation Quickstart")

<Experiment: artifact_location='mlflow-artifacts:/108347913608011064', creation_time=1759897034797, experiment_id='108347913608011064', last_update_time=1759897034797, lifecycle_stage='active', name='GenAI Evaluation Quickstart', tags={}>

Go to http://localhost:2000 to access the mlflow UI

In [2]:
# Load environment variables

import dotenv

dotenv.load_dotenv()

True

In [3]:
# Create prediction function (ensure you have an openai api key)

from openai import OpenAI

client = OpenAI()

def qa_predict_fn(question: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content

In [4]:
qa_predict_fn("What is the capital of France?")

'The capital of France is Paris.'

In [5]:
# Define evaluation dataset

eval_dataset = [
    {
        "inputs": {"question": "What is the capital of France?"},
        "expectations": {"expected_response": "Paris"}
    },
    {
        "inputs": {"question": "What is 2+2?"},
        "expectations": {"expected_response": "4"}
    },
    {
        "inputs": {"question": "What is in the middle of black and white?"},
        "expectations": {"expected_response": "Grey"}
    }
]

In [6]:
# Create a custom scorer

from mlflow.genai import scorer

@scorer
def is_concise(outputs: str) -> bool:
    """Evaluate if the answer is concise (less than 5 words)"""
    return len(outputs.split()) <= 5

In [ ]:
# Define list of scorers

from mlflow.genai.scorers import Correctness, Guidelines # LLM-based scorers

scorers = [
    Correctness(),
    Guidelines(name="english_guidelines", guidelines="The response should be in English."),
    is_concise
]

In [8]:
# Run evaluation

results = mlflow.genai.evaluate(
    data=eval_dataset,
    scorers=scorers,
    predict_fn=qa_predict_fn,
)

2025/10/08 15:19:08 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/10/08 15:19:08 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.
/Users/rajpulapakura/Code/learning/mlflow/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Evaluating: 100%|██████████| 3/3 [Elapsed: 00:03, Remaining: 00:00] 


Go to http://localhost:2000, find the **GenAI Evaluation Quickstart** experiment and view the traces.

In [9]:
# Create a prediction function that will pass the conciseness scorer

def qa_concise_predict_fn(question: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant, who always responds in 5 words or less."},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content

In [10]:
# Evaluate new prediction function with the same scorers and dataset

mlflow.genai.evaluate(
    data=eval_dataset,
    scorers=scorers,
    predict_fn=qa_concise_predict_fn
)

2025/10/08 15:23:22 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.
Evaluating: 100%|██████████| 3/3 [Elapsed: 00:02, Remaining: 00:00] 


EvaluationResult(
  run_id: 935d7fe4256a461f9be8390e70be2968
  metrics:
    is_concise/mean: 1.0
    english_guidelines/mean: 1.0
    correctness/mean: 1.0
  result_df: [3 rows x 18 cols]
)

Go to http://localhost:2000, find the **GenAI Evaluation Quickstart** experiment and view the traces. They should have all passed now.